In [1]:
# Setup

%load_ext autoreload
%autoreload 2

import logging
from pathlib import Path

from comments import prompts
from comments.llm_labeling import LLMLabeler
from comments.static_labeling import label_unfinished_comments, label_code_snippets_comments
from shared import DifferencesEvaluator, load_dataset, save_dataset
from shared.utils import map_labels, dataset_remove_properties
from shared.deduplicate import JsonDeduplicator
from shared.llm_connector import Model
from pydantic import BaseModel
from typing import Any

logging.basicConfig(level=logging.INFO)
deduplicated_file_path = None

In [ ]:
# Deduplication

input_path = "/media/martin/Big data1/datasets/pa165-git/output.json"
deduplicator = JsonDeduplicator()
file_path = Path(input_path)

deduplicated_file_path = deduplicator.deduplicate_dataset_file(file_path)

In [ ]:
# Labeling unfinished comments - LLM

UNFINISHED_COMMENT_ATTR = "unfinished_comment_llm"

if deduplicated_file_path is None:
    deduplicated_file_path = Path(input("Path to deduplicated file: "))

class ResponseModel(BaseModel):
    is_unfinished: bool

def process_response(response: ResponseModel, element: dict[str, Any]):
    element[UNFINISHED_COMMENT_ATTR] = 1 if response.is_unfinished else 0

labeler = LLMLabeler[ResponseModel](init_prompt=prompts.TODO_COMMENTS_INIT_PROMPT, run_prompt=prompts.RUN_PROMPT_1,
                                    labeled_attribute=UNFINISHED_COMMENT_ATTR, model=Model.LLAMA_3_1,
                                    process_response=process_response, response_class=ResponseModel)
await labeler.label_dataset(deduplicated_file_path, "-llama-labeled-unfinished")


In [ ]:
# Labeling unfinished comments - static rules

if deduplicated_file_path is None:
    deduplicated_file_path = Path(input("Path to deduplicated file: "))

label_unfinished_comments(deduplicated_file_path)


In [ ]:
# Labeling unfinished comments - finding differences

if deduplicated_file_path is None:
    deduplicated_file_path = Path(input("Path to deduplicated file: "))

evaluator = DifferencesEvaluator(new_key_name="unfinished_comment_label", to_compare_keys=["unfinished_comment_llm", "unfinished_comment_static"], no_conflict_key="unfinished_comment_static")
dataset = load_dataset(deduplicated_file_path)
evaluator.evaluate(dataset)
save_dataset(deduplicated_file_path, dataset)

In [2]:
# Labeling commented code comments - LLM

CODE_COMMENT_ATTR = "code_comment_llm"

if deduplicated_file_path is None:
    deduplicated_file_path = Path(input("Path to deduplicated file: "))

class ResponseModel(BaseModel):
    is_code_comment: bool

def process_response(response: ResponseModel, element: dict[str, Any]):
    element[CODE_COMMENT_ATTR] = 1 if response.is_code_comment else 0


labeler = LLMLabeler[ResponseModel](init_prompt=prompts.CODE_COMMENTS_INIT_PROMPT_2, run_prompt=prompts.RUN_PROMPT_1,
                                    labeled_attribute=CODE_COMMENT_ATTR, model=Model.LLAMA_3_1,
                                    process_response=process_response, response_class=ResponseModel)
await labeler.label_dataset(deduplicated_file_path, "-llm-labeled")

INFO:LLMLabeler:Total number of elements: 5
DEBUG:OllamaConnector:Clearing context
INFO:LLMLabeler:Total number of elements: 5
DEBUG:OllamaConnector:Clearing context
INFO:LLMLabeler:Total number of elements: 5
DEBUG:OllamaConnector:Clearing context
INFO:LLMLabeler:Total number of elements: 5
DEBUG:OllamaConnector:Clearing context
INFO:LLMLabeler:Total number of elements: 5
DEBUG:OllamaConnector:Clearing context
INFO:LLMLabeler:Total number of elements: 5
DEBUG:OllamaConnector:Clearing context
INFO:LLMLabeler:Total number of elements: 5
DEBUG:OllamaConnector:Clearing context
INFO:LLMLabeler:Total number of elements: 5
DEBUG:OllamaConnector:Clearing context
INFO:LLMLabeler:Total number of elements: 5
DEBUG:OllamaConnector:Clearing context
INFO:LLMLabeler:Total number of elements: 5
DEBUG:OllamaConnector:Clearing context
INFO:LLMLabeler:Total number of elements: 5
DEBUG:OllamaConnector:Clearing context
INFO:LLMLabeler:Total number of elements: 5
DEBUG:OllamaConnector:Clearing context
INFO

PosixPath('/media/martin/Big data1/datasets/github-projects/thingsboard-comments-deduplicated-llm-labeled.json')

In [ ]:
# Labeling commented code comments - static rules

if deduplicated_file_path is None:
    deduplicated_file_path = Path(input("Path to deduplicated file: "))

label_code_snippets_comments(deduplicated_file_path)

In [ ]:
# Labeling commented code comments - finding differences

if deduplicated_file_path is None:
    deduplicated_file_path = Path(input("Path to deduplicated file: "))

evaluator = DifferencesEvaluator(new_key_name="code_comment_label", to_compare_keys=["code_comment_llm", "code_comment_static"], no_conflict_key="code_comment_static")
dataset = load_dataset(deduplicated_file_path)
evaluator.evaluate(dataset)
save_dataset(deduplicated_file_path, dataset)

In [ ]:
# Map LLM a static labels to final label

if deduplicated_file_path is None:
    deduplicated_file_path = Path(input("Path to deduplicated file: "))

mapping = {
    "code_comment_label": "code_comment",
    "unfinished_comment_label": "technical_debt"
}

dataset = load_dataset(deduplicated_file_path)
dataset = map_labels(mapping, "clean", dataset)

save_dataset(deduplicated_file_path, dataset)


In [ ]:
# Remove unused dataset properties

if deduplicated_file_path is None:
    deduplicated_file_path = Path(input("Path to deduplicated file: "))

dataset = load_dataset(deduplicated_file_path)
dataset_remove_properties(["unfinished_comment_llm", "unfinished_comment_static", "unfinished_comment_label", "code_comment_llm", "code_comment_static", "code_comment_label"], dataset)

save_dataset(deduplicated_file_path, dataset)